In [ ]:
import pandas as pd
import numpy as np
import ta
import logging
import joblib
import json
from datetime import datetime, timedelta
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from catboost import CatBoostClassifier
from typing import Dict, Tuple
from sklearn.model_selection import TimeSeriesSplit, learning_curve
from sklearn.metrics import f1_score
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_selection import RFECV
import matplotlib.pyplot as plt
import optuna
import sys
from dateutil.relativedelta import relativedelta
from sklearn.base import BaseEstimator, TransformerMixin


class TypeCaster(BaseEstimator, TransformerMixin):
    """Cast specified columns back to int32 after the ColumnTransformer."""
    def __init__(self, cat_idx):
        self.cat_idx = cat_idx
    def fit(self, X, y=None):
        return self
    def transform(self, X):               
        X = X.copy()
        # 2) force object dtype so CatBoost will accept cat_features
        X = X.astype(object)
        # 3) overwrite only your categorical-feature columns with int codes
        for idx in self.cat_idx:
            # take the entire column, astype to int, then re-assign
            col = X[:, idx].astype(np.int32)
            X[:, idx] = col
        return X
# -------------------- Threshold Load Helper --------------------
def load_threshold(path="gold1h_final_threshold.txt"):
    return float(Path(path).read_text().strip())

# -------------------- Top-Level Function for Pickling --------------------
def to_int(X):
    return X.astype(int)

# -------------------- Helper to Construct Full Feature Vector --------------------
def construct_feature_vector(current_row: pd.Series, feature_order: list):
    features = {}
    for name in feature_order:
        if name in current_row:
            features[name] = current_row[name]
        else:
            if name == 'ema_diff':
                features[name] = current_row["EMA20"] - current_row["EMA200"]
            elif name == 'bb_penetration':
                features[name] = (current_row["Close"] - current_row["BB_upper"]) / current_row["BB_upper"]
            elif name == 'bb_lower_penetration':
                features[name] = (current_row["Close"] - current_row["BB_lower"]) / current_row["BB_lower"]
            elif name == 'bb_width':
                features[name] = (current_row["BB_upper"] - current_row["BB_lower"]) / current_row["Close"]
            elif name == 'atr_ratio':
                features[name] = current_row["ATR"] / current_row["ATR_MA"] if current_row["ATR_MA"] != 0 else 0
            else:
                features[name] = np.nan
    return features

# -------------------- Helper for Risk Factor --------------------
def get_risk_factor(symbol: str) -> float:
    return 0.01 if "1D" in symbol else 0.005

# ==================== CONFIGURATION ====================
class StrategyConfig:
    RISK_PER_TRADE: float = 0.005
    PARTIAL_CLOSE_PCT: float = 0.7
    MAX_DD: float = 0.04  # Daily drawdown threshold
    TARGET_THRESHOLD: float = 0.004  # Threshold for 3-class target

    # Unified instruments for 1H timeframe
    INSTRUMENTS: Dict[str, Dict] = {
       
         'EURUSD.4H': {'file': 'EURUSD_H4_200901020400_202506301200_cleaned.csv', 'pip': 0.0001, 'pip_value': 10},
      
    }

    # Model and scaler paths for unified strategy
    MODEL_PATH = 'index_pure_strategy_US5001Hv11.pkl'
    SCALER_PATH = 'eurusd_4h_v1_scaler.pkl'

    # Training range
    TRAIN_START: datetime = datetime(2018, 1, 1)
    TRAIN_END: datetime = datetime(2024, 12, 31)
# ==================== CORE RULES ====================
class PureStrategy:
    @staticmethod
    def validate_entry(df: pd.DataFrame, idx: int) -> bool:
        r = df.iloc[idx]
        return (
            (r['breakout_flag'] == 1 or r['momentum_flag'] == 1)
            and r['ATR'] > r['ATR_MA']
        )

# ==================== DATA ENGINE ====================
class LeanDataEngine:  
    def __init__(self):  
        self.instruments = self._load_clean_data()  

    def _load_clean_data(self) -> Dict[str, pd.DataFrame]:  
        instruments = {}  
        dtype_spec = {'Open': 'float32', 'High': 'float32', 'Low': 'float32',  
                      'Close': 'float32', 'Volume': 'float32'}  
        for sym, config in StrategyConfig.INSTRUMENTS.items():  
            df = pd.read_csv(config['file'], dtype=dtype_spec)  
            if 'Date' in df.columns:  
                df['Date'] = pd.to_datetime(df['Date'], errors='coerce')  
                df = df.dropna(subset=['Date'])  
                df.set_index('Date', inplace=True)  
            elif 'Datetime' in df.columns:  
                df['Datetime'] = pd.to_datetime(df['Datetime'], errors='coerce')  
                df = df.dropna(subset=['Datetime'])  
                df.set_index('Datetime', inplace=True)  
            df.sort_index(inplace=True)  
            df.index = pd.to_datetime(df.index, errors='coerce')  
            df = self._calculate_core_features(df)  
            instruments[sym] = df  
        return instruments  

    def _calculate_core_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df['EMA20'] = ta.trend.ema_indicator(df['Close'], 20)
        df['EMA50'] = ta.trend.ema_indicator(df['Close'], 50)  # New: 50-period EMA
        df['EMA200'] = ta.trend.ema_indicator(df['Close'], 200)
        # New features based on EMA50
        df['ema50_diff'] = df['EMA20'] - df['EMA50']
        df['EMA50_slope'] = df['EMA50'].diff(5) / df['EMA50'].shift(5)
        
        df['BB_upper'] = ta.volatility.bollinger_hband(df['Close'], 20, 2)
        df['BB_lower'] = ta.volatility.bollinger_lband(df['Close'], 20, 2)
        # Add BB width calculation
        df['BB_width'] = (df['BB_upper'] - df['BB_lower']) / df['Close']
        # Add Lower Band penetration like the upper band one
        df['bb_lower_penetration'] = (df['Close'] - df['BB_lower']) / df['BB_lower']
        # right after you compute BB_upper / BB_lower / BB_width:
        df['bb_penetration'] = (df['Close'] - df['BB_upper']) / df['BB_upper']

        df['ATR'] = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'], 14)
        df['ATR_MA'] = df['ATR'].rolling(20).mean()
        # Removed ATR_median100 from here
        # df['ATR_median100'] = df['ATR'].rolling(100).median()
        

        df['ADX'] = ta.trend.ADXIndicator(df['High'], df['Low'], df['Close'], window=14).adx()
        df['Trend_purity'] = df['Close'].rolling(20).corr(df['EMA20'].shift(1))
        df['EMA200_slope'] = df['EMA200'].diff(10) / df['EMA200'].shift(10)
        df['STD_20'] = df['Close'].rolling(20).std()
        df['Vol_ratio'] = df['ATR'] / df['STD_20']
        macd = ta.trend.MACD(df['Close'])
        df['MACD_hist'] = macd.macd_diff()


        
       

        # ——— Look-ahead fix #2: shifted vol_regime ———
        atr_roll = df['ATR'].rolling(100)
        q75 = atr_roll.quantile(0.75).shift(1)
        q40 = atr_roll.quantile(0.40).shift(1)
        df['vol_regime'] = np.where(
            df['ATR'].shift(1) > q75, 2,
            np.where(df['ATR'].shift(1) > q40, 1, 0)
        )



        df['momentum_flag'] = np.sign(df['Close'].diff(5)) * ((abs(df['Close'].diff(5)) / df['ATR']) > 1).astype(int)
        df['RSI14'] = ta.momentum.rsi(df['Close'], window=14)
        

        # Volume features
        df['Volume_MA20'] = df['Volume'].rolling(20).mean()
        df['volume_ratio'] = df['Volume'] / df['Volume_MA20']




        return df.dropna()

# ==================== TRAINER (UNIFIED MODEL) ====================
class BinaryClassTrainer:
    def __init__(self, de: LeanDataEngine):
        self.data = de.instruments
        self.features, self.targets = self._generate_samples()
        print("\nTarget distribution (long=1 / short=0):")
        print(self.targets.value_counts(normalize=True).rename("pct"))
 

    # ----------------------------------------------------------------------
    #  SAMPLE GENERATION  (unchanged)
    # ----------------------------------------------------------------------
    def _generate_samples(self) -> Tuple[pd.DataFrame, pd.Series]:
        rows: list[dict] = []
        for sym, df in self.data.items():
            for i in range(100, len(df) - 6):
                r   = df.iloc[i]
                fut = df['Close'].iloc[i + 6]
                pct = (fut - r['Close']) / r['Close']
                tgt = 1 if pct > StrategyConfig.TARGET_THRESHOLD else 0
                rows.append({
                    'datetime': r.name,
                    'ema_diff': r['EMA20'] - r['EMA200'],
                    'bb_penetration': (r['Close'] - r['BB_upper']) / r['BB_upper'],
                    'bb_lower_penetration': r['bb_lower_penetration'],
                    'bb_width': r['BB_width'],
                    'atr_ratio': r['ATR'] / r['ATR_MA'],
                    'adx': r['ADX'],
                    'macd_hist': r['MACD_hist'],
                    'ema200_slope': r['EMA200_slope'],
                    'vol_ratio': r['Vol_ratio'],
                    'trend_purity': r['Trend_purity'],
                    'ema50_diff': r['ema50_diff'],
                    'EMA50_slope': r['EMA50_slope'],
                    'vol_regime': r['vol_regime'],
                    'momentum_flag': r['momentum_flag'],
                    'rsi14': r['RSI14'],
                    'volume_ratio': r['volume_ratio'],
                    'target': tgt
                })
        df = pd.DataFrame(rows).set_index('datetime')
        return df.drop(columns='target'), df['target']

    # ----------------------------------------------------------------------
    #  MODEL TRAINING  (categoricals preserved – no other changes)
    # ----------------------------------------------------------------------
    def train_model(self, best_params: dict = None) -> Pipeline:
        # feature groups
        original_features   = ['ema_diff', 'bb_penetration', 'bb_lower_penetration', 'bb_width', 'atr_ratio']
        new_features        = ['adx', 'rsi14' , 'ema200_slope', 'vol_ratio', 'trend_purity', 'ema50_diff', 'EMA50_slope']
        momentum_features = ['macd_hist', 'momentum_flag']

        volume_features     = ['volume_ratio']

        all_features = original_features + new_features  + momentum_features  + volume_features
        self.features = self.features[all_features]

        # preprocessing
        preprocessor = ColumnTransformer(
            [
                ('robust',            RobustScaler(),  original_features),
                ('standard',          StandardScaler(), new_features),
            ],
            remainder='passthrough',
            sparse_threshold=0
        )

        
        cat_idx = []


        # 4) -----------------------------------------------------------------
        # CatBoost classifier
        model = Pipeline([
            ('preprocessor', preprocessor),
            ('typecaster',  TypeCaster(cat_idx=cat_idx)),
            ('cat',  CatBoostClassifier(
                iterations     = 1000,
                depth          = 3,
                learning_rate  = 0.030,
                bootstrap_type='Bernoulli',
                loss_function  = 'Logloss',
                subsample=0.60, 
                l2_leaf_reg=120,
                border_count=101,
                thread_count   = -1,
                verbose        = False,
                cat_features   = cat_idx
            )),
        ])
       
        if best_params:
            model.named_steps['cat'].set_params(**best_params)


        # 6) -----------------------------------------------------------------
        # Class weighting & fit
        classes = np.unique(self.targets)
        cw = compute_class_weight('balanced', classes=classes, y=self.targets)
        sample_weights = np.array([cw[int(y)] for y in self.targets])

        model.fit(self.features, self.targets, cat__sample_weight=sample_weights)
        joblib.dump(model, StrategyConfig.MODEL_PATH)
        joblib.dump(model.named_steps['preprocessor'], StrategyConfig.SCALER_PATH)

        # 7) -----------------------------------------------------------------
        # Learning-curve snapshot
        self.plot_learning_curve(model, self.features, self.targets)

        return model

    def plot_learning_curve(self, model, X, y):
        """
        Visualize per-fold and mean learning curves, save with timestamped filename.
        """
        cv = TimeSeriesSplit(n_splits=5)
        train_sizes, train_scores, test_scores = learning_curve(
            estimator=model,
            X=X,
            y=y,
            cv=cv,
            scoring='f1_macro',
            train_sizes=np.linspace(0.1, 1.0, 5),
            n_jobs=-1,
            verbose=0
        )

        plt.figure(figsize=(10, 6))
        # plot each fold
        for fold in range(train_scores.shape[1]):
            plt.plot(
                train_sizes,
                train_scores[:, fold],
                '--', alpha=0.3,
                label=f"Train fold {fold+1}"
            )
            plt.plot(
                train_sizes,
                test_scores[:, fold],
                '--', alpha=0.3,
                label=f"CV fold {fold+1}"
            )

        # plot means
        plt.plot(
            train_sizes,
            np.mean(train_scores, axis=1),
            'o-', linewidth=2,
            label="Mean Training F1"
        )
        plt.plot(
            train_sizes,
            np.mean(test_scores, axis=1),
            'o-', linewidth=2,
            label="Mean CV F1"
        )

        plt.title("index 1h chart Learning Curve (per-fold & mean)")
        plt.xlabel("Training Examples")
        plt.ylabel("F1 Score")
        plt.legend(loc="best", ncol=2, fontsize="small")
        plt.grid(True)
        plt.tight_layout()

        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        fname = f"eurusd4hchartv1_learning_curve_{ts}.png"
        plt.savefig(fname, bbox_inches='tight')
        print(f"✅ Saved learning curve with folds: {fname}")
        plt.show()

# ==================== OPTIMIZER ====================
class PureOptimizer:
    def __init__(self, trainer: BinaryClassTrainer):
        self.features = trainer.features
        self.targets  = trainer.targets

    def optimize(self) -> Dict:
        def objective(trial: optuna.Trial):
            params = {
                'iterations':      trial.suggest_int('iterations', 500, 800),
                'depth':           trial.suggest_int('depth', 3, 7),
                'learning_rate':   trial.suggest_float('learning_rate', 0.005, 0.015),
                'l2_leaf_reg':     trial.suggest_float('l2_leaf_reg', 600.0, 700.0),
                'random_strength': trial.suggest_float('random_strength', 10.0, 30.0),
                'subsample':       trial.suggest_float('subsample', 0.5, 1.0),
                'border_count':    trial.suggest_int('border_count', 64, 128),
                'min_data_in_leaf'  : trial.suggest_int('min_data_in_leaf', 70, 90),
                'rsm'               : trial.suggest_float('rsm', 0.6, 0.8),
            }
            tscv = TimeSeriesSplit(n_splits=5)
            scores = []
            for fold_num, (train_idx, test_idx) in enumerate(tscv.split(self.features)):  
                model = CatBoostClassifier(
                    bootstrap_type='Bernoulli',
                    **params,  
                    loss_function='Logloss',  
                    thread_count=-1,  
                    verbose=False  
                )  

                classes = np.unique(self.targets.iloc[train_idx])  
                cw = compute_class_weight('balanced', classes=classes, y=self.targets.iloc[train_idx])  
                sw = np.array([cw[int(y)] for y in self.targets.iloc[train_idx]])  

                model.fit(  
                    self.features.iloc[train_idx],  
                    self.targets.iloc[train_idx],  
                    sample_weight=sw 
                )  

                preds = model.predict(self.features.iloc[test_idx])  
                score = f1_score(self.targets.iloc[test_idx], preds, average='macro')  
                scores.append(score)  

                trial.report(score, fold_num)  
                if trial.should_prune():  
                    raise optuna.TrialPruned()  

            return np.mean(scores)  

        study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())  
        study.optimize(objective, n_trials=100, n_jobs=1)  # Reduced trials
        return study.best_params

# ==================== PERFORMANCE REPORT ====================
class PerformanceReport:
    def __init__(self, engine):
        self.trades = [t for t in engine.trades if t['status'] == 'closed']
        self.equity = engine.equity
        self.equity_timeseries = engine.equity_timeseries

    def _compute_sharpe(self):    
        if self.equity_timeseries.empty:    
            return 0    
        daily_equity = self.equity_timeseries.resample('D').last().ffill()    
        returns = daily_equity.pct_change().dropna()    
        if returns.std() == 0:    
            return 0    
        sharpe = (returns.mean() / returns.std()) * np.sqrt(252)    
        return np.clip(sharpe, -5, 5)    

    def _compute_profit_factor(self):    
        total_profit = 0    
        total_loss = 0    
        for trade in self.trades:    
            if trade['direction'] == 'buy':    
                pnl = (trade['exit_price'] - trade['entry_price']) * trade['size'] * trade['config']['pip']    
            else:    
                pnl = (trade['entry_price'] - trade['exit_price']) * trade['size'] * trade['config']['pip']    
            if pnl > 0:    
                total_profit += pnl    
            elif pnl < 0:    
                total_loss += abs(pnl)    
        return total_profit / total_loss if total_loss > 0 else float('inf')    

    def generate(self):    
        total_return = (self.equity[-1] / 100000.0 - 1)    
        peaks = np.maximum.accumulate(self.equity)    
        dd = np.max((peaks - self.equity) / peaks)    
        wins = 0    
        for t in self.trades:    
            if t['direction'] == 'buy' and t['exit_price'] > t['entry_price']:    
                wins += 1    
            elif t['direction'] == 'sell' and t['exit_price'] < t['entry_price']:    
                wins += 1    
        total = len(self.trades)    
        win_rate = wins / total if total > 0 else 0    
        sharpe = self._compute_sharpe()    
        profit_factor = self._compute_profit_factor()    
        buy_trades = len([t for t in self.trades if t['direction'] == 'buy'])    
        sell_trades = len([t for t in self.trades if t['direction'] == 'sell'])    
        print(f"\n{' BACKTEST RESULTS ':=^40}")    
        print(f"Initial Capital: $100,000.00")    
        print(f"Final Equity: ${self.equity[-1]:,.2f}")    
        print(f"Total Return: {total_return:.1%}")    
        print(f"Max Drawdown: {dd:.1%}")    
        print(f"Total Trades: {total}")    
        print(f"Buy Trades: {buy_trades}")    
        print(f"Sell Trades: {sell_trades}")    
        print(f"Win Rate: {win_rate:.1%}")    
        print(f"Sharpe Ratio: {sharpe:.2f}")    
        print(f"Profit Factor: {profit_factor:.2f}")    
        self._plot_equity()    

    def _plot_equity(self):    
        plt.figure(figsize=(10, 6))    
        plt.plot(self.equity)    
        plt.title("Equity Curve")    
        plt.xlabel("Time")    
        plt.ylabel("Equity ($)")    
        plt.grid(True)    
        plt.show()

# ==================== BACKTEST DATA LOADER ====================
class BacktestDataLoader:
    def __init__(self):
        self.bull_model = None
        self.feature_order = None
        data_engine = LeanDataEngine()
        self.instruments = {}
        for sym, df in data_engine.instruments.items():
            self.instruments[sym] = {
                'data': df,
                'config': StrategyConfig.INSTRUMENTS[sym]
            }
        print("Backtest data loaded.")

# ==================== EXECUTION ENGINE ====================
class ExecutionEngine:
    def __init__(self):
        self.trades = []
        self.equity = [100000.0]
        self.open_trades = 0
        self.cooldowns = {}
        self.current_day = None
        self.daily_equity_start = None
        self.equity_timeseries = pd.Series([100000.0], index=[pd.Timestamp(datetime.now().date())])
        self.trade_counts = {"US30.cash": 0, "US100.cash": 0, "US500.cash": 0, "GER40.cash": 0, "UK100.cash": 0}
        self.buy_count = 0
        self.sell_count = 0

    def update_equity_daily(self, current_dt: datetime):    
        current_date = current_dt.date()    
        last_date = self.equity_timeseries.index[-1].date()    
        if current_date > last_date:    
            last_equity = self.equity[-1]    
            new_entry = pd.Series([last_equity], index=[pd.Timestamp(current_date)])    
            self.equity_timeseries = pd.concat([self.equity_timeseries, new_entry])    

    def execute_trade(self, signal: dict):    
        if any(t['symbol'] == signal['symbol'] and t['status'] == 'open' for t in self.trades):    
            return    

        current_day = signal['datetime'].date()    
        if self.current_day != current_day:    
            self.current_day = current_day    
            self.daily_equity_start = self.equity[-1]    
        elif self.equity[-1] < self.daily_equity_start * (1 - StrategyConfig.MAX_DD):    
            print(f"Daily drawdown reached on {signal['datetime'].date()}. No new trade for {signal['symbol']}.")    
            return    

        current_equity = self.equity[-1]    
        total_risk = 0.0    
        for trade in self.trades:    
            if trade['status'] == 'open':    
                if trade['direction'] == 'buy':    
                    risk_trade = (trade['entry_price'] - trade['sl']) * trade['size'] * trade['config']['pip']    
                else:    
                    risk_trade = (trade['sl'] - trade['entry_price']) * trade['size'] * trade['config']['pip']    
                total_risk += risk_trade    
        risk_percent = total_risk / current_equity    
        if risk_percent >= 0.03:    
            print(f"Current risk {risk_percent:.2%} >= 3%. No new trade for {signal['symbol']}.")    
            return    

        risk = StrategyConfig.RISK_PER_TRADE    
        risk_amount = self.equity[-1] * risk    
        pip_value = signal['config'].get('pip_value', signal['config']['pip'])    
        size = risk_amount / (2 * signal['atr'] * pip_value)    
        size = max(size, 0.01)    
        entry_price = signal['price']    

        if signal['direction'] == 'buy':    
            tp = entry_price + 3 * signal['atr']    
            sl = entry_price - 2 * signal['atr']    
        elif signal['direction'] == 'sell':    
            tp = entry_price - 3 * signal['atr']    
            sl = entry_price + 2 * signal['atr']    

        trade = {    
            'symbol': signal['symbol'],    
            'entry_time': signal['datetime'],    
            'entry_price': entry_price,    
            'size': size,    
            'direction': signal['direction'],    
            'tp': tp,    
            'sl': sl,    
            'status': 'open',    
            'config': signal['config']    
        }    
        self.trades.append(trade)    
        self.open_trades += 1    

        sym = signal['symbol']    
        self.trade_counts[sym.split('.')[0] + '.cash'] = self.trade_counts.get(sym.split('.')[0] + '.cash', 0) + 1    
        if signal['direction'] == 'buy':    
            self.buy_count += 1    
        elif signal['direction'] == 'sell':    
            self.sell_count += 1    

    def _partial_close(self, trade: dict, price: float):    
        partial_size = trade['size'] * StrategyConfig.PARTIAL_CLOSE_PCT    
        if trade['direction'] == 'buy':    
            profit = partial_size * (price - trade['entry_price']) * trade['config']['pip']    
        else:    
            profit = partial_size * (trade['entry_price'] - price) * trade['config']['pip']    
        new_equity = self.equity[-1] + profit    
        self.equity.append(new_equity)    
        trade['size'] *= (1 - StrategyConfig.PARTIAL_CLOSE_PCT)    
        trade['partial_closed'] = True    

    def _close_trade(self, trade: dict, price: float, reason: str, current_dt: datetime):    
        if trade['direction'] == 'buy':    
            profit = trade['size'] * (price - trade['entry_price']) * trade['config']['pip']    
        else:    
            profit = trade['size'] * (trade['entry_price'] - price) * trade['config']['pip']    
        new_equity = self.equity[-1] + profit    
        self.equity.append(new_equity)    
        trade['exit_price'] = price    
        trade['exit_reason'] = reason    
        trade['exit_time'] = current_dt    
        trade['status'] = 'closed'    
        self.open_trades -= 1    
        if reason in ['TP', 'SL']:    
            self.cooldowns[trade['symbol']] = current_dt    
        self.equity_timeseries[current_dt] = new_equity    

    def manage_trades(self, current_dt: datetime, price_data: dict, model, feature_order: list):    
        self.update_equity_daily(current_dt)    
        for trade in self.trades:    
            if trade['status'] != 'open':    
                continue    
            symbol = trade['symbol']    
            symbol_df = price_data[symbol]    
            current_row = symbol_df.asof(current_dt)    
            if current_row is None or pd.isna(current_row['Close']):    
                continue    
            current_price = current_row['Close']    
            progress = abs(current_price - trade['entry_price']) / (abs(trade['tp'] - trade['entry_price']) + 1e-8)    
            if progress >= StrategyConfig.PARTIAL_CLOSE_PCT and not trade.get('partial_closed', False):    
                self._partial_close(trade, current_price)    
            exit_reason = None    
            if trade['direction'] == 'buy':    
                if current_price >= trade['tp']:    
                    exit_reason = 'TP'    
                elif current_price <= trade['sl']:    
                    exit_reason = 'SL'    
            else:    
                if current_price <= trade['tp']:    
                    exit_reason = 'TP'    
                elif current_price >= trade['sl']:    
                    exit_reason = 'SL'    
            if exit_reason:    
                self._close_trade(trade, current_price, exit_reason, current_dt)

# ==================== STRATEGY BACKTEST ====================
class StrategyBacktest:
    def __init__(self):
        self.data_loader = None
        self.engine = None
        self.report = None
        self.start_date = None
        self.end_date = None

    def run(self):    
        all_price_data = {sym: self.data_loader.instruments[sym]['data']    
                          for sym in self.data_loader.instruments}    
        for sym in self.data_loader.instruments:    
            df = self.data_loader.instruments[sym]['data']    
            config = self.data_loader.instruments[sym]['config']    
            df = df.loc[self.start_date:self.end_date]    
            for idx in range(len(df)):    
                current_dt = df.index[idx]    
                if PureStrategy.validate_entry(df, idx):    
                    self._process_signal(df, idx, sym, config)    
                self.engine.manage_trades(current_dt, all_price_data, self.data_loader.bull_model, self.data_loader.feature_order)    
        self.report = PerformanceReport(self.engine)    
        self.report.generate()    

    def _process_signal(self, df: pd.DataFrame, idx: int, sym: str, config: dict):    
        features = {    
            'ema_diff': df['EMA20'].iloc[idx] - df['EMA200'].iloc[idx],    
            'bb_penetration': (df['Close'].iloc[idx] - df['BB_upper'].iloc[idx]) / df['BB_upper'].iloc[idx],    
            'bb_lower_penetration': (df['Close'].iloc[idx] - df['BB_lower'].iloc[idx]) / df['BB_lower'].iloc[idx],  
            'bb_width': (df['BB_upper'].iloc[idx] - df['BB_lower'].iloc[idx]) / df['Close'].iloc[idx],  
            'atr_ratio': df['ATR'].iloc[idx] / df['ATR_MA'].iloc[idx],    
            'adx': df['ADX'].iloc[idx],    
            'macd_hist': df['MACD_hist'].iloc[idx],    
            'ema200_slope': df['EMA200_slope'].iloc[idx],    
            'vol_ratio': df['Vol_ratio'].iloc[idx],    
            'trend_purity': df['Trend_purity'].iloc[idx],    
            'ema50_diff': df['ema50_diff'].iloc[idx],    
            'EMA50_slope': df['EMA50_slope'].iloc[idx],    
            'hour_sin': df['hour_sin'].iloc[idx],    
            'hour_cos': df['hour_cos'].iloc[idx],    
            'day_part': df['day_part'].iloc[idx],    
            'london_open': df['london_open'].iloc[idx],    
            'vol_regime': df['vol_regime'].iloc[idx],    
            'ny_session': df['ny_session'].iloc[idx],    
            'asia_session': df['asia_session'].iloc[idx],    
            'breakout_flag':df['breakout_flag'].iloc[idx],      
            'momentum_flag':df['momentum_flag'].iloc[idx],    
            'bb_prox_upper':df['bb_prox_upper'].iloc[idx],    
            'bb_prox_lower':df['bb_prox_lower'].iloc[idx],    
            'volume_ratio': df['volume_ratio'].iloc[idx],
            'volume_spike': df['volume_spike'].iloc[idx]
            
        }    
        feature_order = self.data_loader.feature_order    
        X = pd.DataFrame({name: [features[name]] for name in feature_order}, columns=feature_order)    
        for col in ['vol_regime', 'ny_session', 'asia_session']:    
            X[col] = X[col].astype(int)        
        probs = self.data_loader.bull_model.predict_proba(X)[0]    
        threshold = 0.75    
        if np.max(probs) < 0.6:    
            return    
        if (probs[1] > threshold):  
            direction = 'buy'    
        elif (probs[0] > threshold):    
            direction = 'sell'    
        else:    
            return    
        risk = StrategyConfig.RISK_PER_TRADE    
        signal = {    
            'symbol': sym,    
            'datetime': df.index[idx],    
            'price': df['Close'].iloc[idx],    
            'atr': df['ATR'].iloc[idx],      
            'config': config,    
            'direction': direction,    
            'risk': risk    
        }    
        self.engine.execute_trade(signal)    

    def run_single_training_and_backtests(self):    
        print("\n=== Training Unified Model: 2016 to Jan 2025 ===")    
        data_engine = LeanDataEngine()    
        trainer = BinaryClassTrainer(data_engine)    
        optimizer = PureOptimizer(trainer)    
        best_params = optimizer.optimize()    
        print("Best hyperparameters from Bayesian optimization:", best_params)    
        with open("best_params_index_gold1hv4.json", "w") as fp:    
            json.dump(best_params, fp, indent=4)    
        print("Best parameters saved to best_params_index_gold1hv4.json")    
        model = trainer.train_model(best_params=best_params)    
        joblib.dump(model, StrategyConfig.MODEL_PATH)    
        joblib.dump(model.named_steps['preprocessor'], StrategyConfig.SCALER_PATH)    
        print(f"✅ Unified model saved: {StrategyConfig.MODEL_PATH}, {StrategyConfig.SCALER_PATH}")    
        loader = BacktestDataLoader()    
        loader.bull_model = model    
        loader.feature_order = model.named_steps['preprocessor'].feature_names_in_    
        self.data_loader = loader    

        print("\n=== Backtest: Jan 2017 to Apr 2025 ===")    
        self.start_date = datetime(2017, 1, 1)    
        self.end_date = datetime(2025, 4, 11)    
        self.engine = ExecutionEngine()    
        self.run()

# ==================== WALK-FORWARD WINDOW GENERATOR ====================
class WalkForwardWindows:
    def __init__(self, train_start=datetime(2018, 1, 1)):
        self.train_start = train_start

    def generate_windows(self):    
        windows = []    
        # Yearly folds 2018/19 to 2023/24
        for yr in range(self.train_start.year, 2024):
            windows.append({
                "train_start": self.train_start,
                "train_end":   datetime(yr, 12, 31),
                "test_start":  datetime(yr + 1, 1, 1),
                "test_end":    datetime(yr + 1, 12, 31)
            })
        # Final fold - train through 2024, test 2025
        windows.append({
            "train_start": self.train_start,
            "train_end":   datetime(2024, 12, 31),
            "test_start":  datetime(2025, 1, 1),
            "test_end":    datetime(2025, 4, 11)
        })
        return windows


# ==================== WALK-FORWARD HELPER FUNCTIONS ====================
def slice_engine(base_engine: LeanDataEngine,
                 start: datetime, end: datetime) -> LeanDataEngine:
    """Return a LeanDataEngine clone with each DataFrame cut to [start:end]."""
    clone = LeanDataEngine.__new__(LeanDataEngine)
    clone.instruments = {s: df.loc[start:end]
                         for s, df in base_engine.instruments.items()}
    return clone

def best_threshold(probs: np.ndarray, y_true: pd.Series,
                   lo=0.60, hi=0.95, step=0.01):
    """Scan probability cut-offs and return (best_threshold, best_F1)."""
    best_t, best_f1 = lo, -1
    for t in np.arange(lo, hi + 1e-9, step):
        f1 = f1_score(y_true, probs[:, 1] > t, average="macro")
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

# ==================== WALK-FORWARD RUNNER ====================
class WalkForwardRunner:
    """Run walk-forward tuning, then train one final model on all data."""
    def __init__(self):
        # Load full dataset once
        self.base_engine = LeanDataEngine()
        # Build full feature/target frames for slicing
        full_trainer       = BinaryClassTrainer(self.base_engine)
        self.full_X, self.full_y = full_trainer.features, full_trainer.targets
        # Prepare windows
        self.windows        = WalkForwardWindows().generate_windows()
        self.fold_params    = []
        self.fold_thresh    = []

    def run(self):
        for i, win in enumerate(self.windows, 1):
            ts, te = win["train_start"], win["train_end"]
            vs, ve = win["test_start"],  win["test_end"]
            print(f"\n=== Fold {i}: train {ts.date()}–{te.date()} → test {vs.date()}–{ve.date()} ===")

            # 1) Train slice via slice_engine
            eng_train = slice_engine(self.base_engine, ts, te)
            n_bars = len(eng_train.instruments['EURUSD.4H'])
            print(f"Fold {i} slice size: {n_bars} bars (need ≥205)")


            trainer   = BinaryClassTrainer(eng_train)
            best_p    = PureOptimizer(trainer).optimize()
            model     = trainer.train_model(best_params=best_p)

            # 2) Slice test from the full feature set
            X_test = self.full_X.loc[vs:ve]
            y_test = self.full_y.loc[vs:ve]
            if X_test.empty:
                print(f"→ No test data in {vs.date()}–{ve.date()}, skipping fold {i}.")
                continue

            # 3) Calibrate probability threshold
            probs    = model.predict_proba(X_test)
            t_star, f1_star = best_threshold(probs, y_test)
            print(f"      F1 = {f1_star:.3f} @ prob > {t_star:.2f}")

            self.fold_params.append(best_p)
            self.fold_thresh.append(t_star)

        # 4) Aggregate median parameters and threshold
        median_params = pd.DataFrame(self.fold_params).median().to_dict()
        for k in ('iterations','depth','border_count'):
            median_params[k] = int(round(median_params[k]))
        median_thresh = float(np.median(self.fold_thresh))

        print("\n=== Median hyper-parameters ===")
        print(json.dumps(median_params, indent=2))
        print(f"Median probability threshold = {median_thresh:.2f}")

        # 5) Final model on all data
        final_trainer = BinaryClassTrainer(self.base_engine)
        final_model   = final_trainer.train_model(best_params=median_params)
        joblib.dump(final_model, "eurusd4h_final_wfv1.pkl")
        with open("eurusd4h_final_thresholdv1.txt", "w") as f:
            f.write(str(median_thresh))
        print("\n✅  Saved gold1h_final_wfv3.pkl and gold1h_final_threshold.txt")


# ==================== MAIN EXECUTION ====================
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    print("Starting Strategy Training")
    
    # Run walk-forward validation
    wf_runner = WalkForwardRunner()
    wf_runner.run()
    
    print("\nTraining completed successfully!")